# Day 3 — EDA + Mapping

**Estimated time:** 2:00 PM – 5:00 PM  •  **Prereq:** Day 2 checkpoints written

Yesterday you cleaned the NTD ridership file and the messy station
frequencies CSV. Today you turn those tables into **insight and
pictures.**

Three sections:

1. **Exploratory Data Analysis (EDA)** — `.describe()`, `.groupby()`,
   `.value_counts()`. Small vocabulary, enormous leverage.
2. **Map the rail network with Folium.** Real interactive HTML map —
   the kind a decision-maker would actually look at.
3. **Overlay World Cup venues.** Measure stadium-to-station distances
   and save the Day 3 checkpoint.

## Learning objectives

- [ ] Use the three most important pandas summary verbs.
- [ ] Build a Folium map with multiple layers (lines, stations, venues).
- [ ] Measure great-circle distance between two lat/lon points.
- [ ] Save `data/checkpoints/day3_output_eda_summary.csv` for Day 4.


## 1. Setup and data load


In [ ]:
# Setup — run this first. In Colab: clones the repo and installs deps.
# Locally: no-op (the conda env handles it).
import os, sys, urllib.request
try:
    import google.colab  # noqa: F401
    if not os.path.exists('/tmp/colab_bootstrap.py'):
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/arctic-gsu/arctic-scisynth-2026-rwd/main/setup/colab_bootstrap.py',
            '/tmp/colab_bootstrap.py',
        )
    sys.path.insert(0, '/tmp')
    from colab_bootstrap import bootstrap
    bootstrap()
except ImportError:
    pass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data' / 'raw').is_dir():
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
CHECKPOINTS = ROOT / 'data' / 'checkpoints'

MARTA_NTD_ID = 40022
MODE_NAMES = {'HR': 'Heavy Rail', 'MB': 'Bus', 'SR': 'Streetcar',
              'DR': 'Demand Response'}
STADIUM_ADJACENT = ['sec district', 'vine city']
STADIUM_AREA = STADIUM_ADJACENT + ['five points']

# World Cup demand model (Day 4). Canonical values; sources cited in
# the Day 4 notebook narrative. Changing any value here propagates
# through Day 4 and Day 5 with no duplicate literals to update.
STADIUM_CAPACITY = 75_000
MATCH_DAY_TRAINS_PER_HOUR = 24
ARRIVAL_WINDOW = 2.5
MODE_SHARE = {'low': 0.15, 'central': 0.22, 'high': 0.35}
TRAIN_CAP = {'seated_only': 232, 'service_standard': 400, 'full_crush': 750}
MS_ORDER = ('low', 'central', 'high')
CAP_ORDER = ('seated_only', 'service_standard', 'full_crush')
# GT regression (Santanam et al. 2021, arXiv:2106.05359; MAPE 11.7%).
GT_INTERCEPT = -1201
GT_SLOPE = 0.1739

print(f'✅ Imports ready. Working from {ROOT}')


Load the two Day 2 checkpoints. If you didn't finish Day 2 in time,
don't worry — these files are **committed in the repo**, so this works
even if your own `day2_output_*.csv` never got written.


In [ ]:
ridership = pd.read_csv(
    CHECKPOINTS / 'day2_output_cleaned_ridership.csv', parse_dates=['date']
)
station_freq = pd.read_csv(CHECKPOINTS / 'day2_output_station_freq_clean.csv')
print(f'ridership:    {ridership.shape[0]:,} rows, columns = {list(ridership.columns)}')
print(f'station_freq: {station_freq.shape[0]:,} rows, columns = {list(station_freq.columns)}')


### ✅ Checkpoint


In [ ]:
assert len(ridership) > 100, f'Expected >100 ridership rows; got {len(ridership)}.'
assert {'mode', 'mode_name', 'upt', 'date'}.issubset(ridership.columns), (
    'ridership should have mode, mode_name, upt, date columns.'
)
assert len(station_freq) >= 38, f'Expected ≥38 stations; got {len(station_freq)}.'
assert {'station_name', 'line', 'peak_trains_per_hour', 'lat', 'lon'}.issubset(
    station_freq.columns
), 'station_freq is missing expected columns.'
print(f'✅ Loaded {len(ridership):,} ridership rows and {len(station_freq)} stations.')


## 2. Section 1 — EDA: the three verbs

Three pandas methods solve 80% of exploratory analysis:

- `.describe()` — summary statistics for numeric columns
- `.groupby().agg()` — split-apply-combine (totals by category)
- `.value_counts()` — how many of each label

Type each one by hand. Don't copy from the hint block; your fingers
need the practice.


### 🎯 Exercise 1 — `.describe()` on UPT

What does the distribution of monthly unlinked passenger trips look
like, across every mode and every month in the file? Pick one numeric
column and describe it.


In [ ]:
# 🎯 Your Turn! — describe the 'upt' column.
# Hint: ridership['upt'].describe()
???


<details><summary>💡 Click for solution</summary>

```python
ridership['upt'].describe().round(0)
```

The `.round(0)` makes the output readable — millions of trips don't
need decimal precision.
</details>


### 🎯 Exercise 2 — ridership by mode

Group the ridership DataFrame by `mode_name` and compute the **total**
UPT for each mode across the entire time period. Which mode dominates?


In [ ]:
# 🎯 Your Turn! — total UPT by mode, sorted largest first.
# Hint: ridership.groupby('mode_name')['upt'].sum().sort_values(ascending=False)
???


<details><summary>💡 Click for solution</summary>

```python
ridership.groupby('mode_name')['upt'].sum().sort_values(ascending=False).round(0)
```

Rail and bus are *neck-and-neck* in total ridership — roughly
1.5 billion trips each over the file's decade-plus history. Zoom in
to 2023–present and bus edges ahead of rail (rail's faregate issues
drag its recent numbers down). That's why this question is
interesting: the World Cup spike doesn't arrive on a mode that's
clearly bigger than bus — it arrives on a mode whose ridership is
currently noisy and under repair.
</details>


### 🎯 Exercise 3 — line assignments

Each rail station sits on one or more of the four lines (Red, Gold,
Blue, Green). How many stations per line? Use `.value_counts()` on the
`line` column of `station_freq`.


In [ ]:
# 🎯 Your Turn! — count stations per line.
???


<details><summary>💡 Click for solution</summary>

```python
station_freq['line'].value_counts()
```

The total sums to **exactly 38** — one row per station, with a
single primary line assigned. GOLD is the biggest line. Transfer
stations like Five Points aren't double-counted here; the
pre-processed `station_freq` table gives each station one primary
line label.
</details>


### ✅ Checkpoint — EDA summary

Build a tidy summary table of ridership by mode and save it as
`data/checkpoints/day3_output_eda_summary.csv`. Day 4 depends on this.


In [ ]:
# Build a per-mode summary. The total_upt and mean_monthly columns
# will be used in tomorrow's briefing numbers.
mode_summary = ridership.groupby('mode_name').agg(
    total_upt=('upt', 'sum'),
    mean_monthly=('upt', 'mean'),
    max_monthly=('upt', 'max'),
).round(0)
print(mode_summary.to_string())

eda_summary = mode_summary.reset_index()
# 🎯 Your Turn! Save eda_summary to CHECKPOINTS / 'day3_output_eda_summary.csv'.
???
print('Wrote day3_output_eda_summary.csv')


In [ ]:
out = CHECKPOINTS / 'day3_output_eda_summary.csv'
assert out.exists(), f'❌ {out} was not written.'
check = pd.read_csv(out)
assert len(check) >= 3, f'Expected ≥3 mode rows; got {len(check)}.'
assert 'total_upt' in check.columns, '❌ total_upt column missing.'
print(f'✅ {out.name} written with {len(check)} mode rows.')


## 🔄 Recovery point — end of Section 1

If you got lost above, run the next cell. Section 2 only needs the
station frequencies — the minimum-viable recovery.


In [ ]:
station_freq = pd.read_csv(CHECKPOINTS / 'day2_output_station_freq_clean.csv')
print('✅ Recovery loaded.')


## 3. Section 2 — Map the rail network

Folium produces a real interactive HTML map — the same tool you'd use
on the job if your boss asked for "a map of MARTA rail." We'll add
three layers:

1. **Rail lines** as colored polylines (real MARTA brand colors from
   the GTFS `shapes.txt` file).
2. **Stations** as circle markers, sized by peak trains per hour.
3. **World Cup venues** (stadium, fan fest, hotel clusters) as pin
   icons.

The cell below is the longest of the week. Read through it first, then
run. The comments explain what each block is doing.


In [ ]:
# Folium map — rail lines + stations + venues.
import folium, zipfile

# Official MARTA brand hex, straight from the GTFS routes.txt file.
LINE_COLORS = {
    'BLUE':  '#0075B2',
    'GOLD':  '#D4A723',
    'GREEN': '#009D4B',
    'RED':   '#CE242B',
}

# All three GTFS tables we need, read straight into pandas.
with zipfile.ZipFile(RAW / 'marta_gtfs.zip') as zf:
    shapes = pd.read_csv(zf.open('shapes.txt'))
    trips  = pd.read_csv(zf.open('trips.txt'))
    routes = pd.read_csv(zf.open('routes.txt'))

# Map shape_id → line name. Filter to rail (route_type == 1); unlike
# the location_type gotcha from Day 1, route_type is fully populated.
rail_routes = routes.loc[routes['route_type'] == 1, ['route_id', 'route_short_name']]
shape_to_line = (
    trips[['shape_id', 'route_id']].dropna().drop_duplicates()
    .merge(rail_routes, on='route_id')
    .set_index('shape_id')['route_short_name']
)
rail_shapes = shapes[shapes['shape_id'].isin(shape_to_line.index)]

# Pick the longest shape per line — the end-to-end path.
shape_per_line = (
    rail_shapes.groupby('shape_id').size().to_frame('n_points')
    .join(shape_to_line.rename('line'))
    .sort_values('n_points', ascending=False).drop_duplicates('line')
)

wc_ref = pd.read_csv(PROCESSED / 'worldcup_reference.csv')
stadium = wc_ref[wc_ref['type'] == 'stadium'].iloc[0]
m = folium.Map(location=[stadium['lat'], stadium['lon']],
               zoom_start=11, tiles='CartoDB positron')

# Layer 1 — rail lines
for sid, row in shape_per_line.iterrows():
    pts = rail_shapes[rail_shapes['shape_id'] == sid].sort_values('shape_pt_sequence')
    coords = list(zip(pts['shape_pt_lat'].iloc[::10],
                      pts['shape_pt_lon'].iloc[::10]))
    folium.PolyLine(coords, color=LINE_COLORS[row['line']],
                    weight=4, opacity=0.85,
                    tooltip=f"{row['line']} Line").add_to(m)

# Layer 2 — stations, circle radius ∝ peak trains/hour
for _, s in station_freq.iterrows():
    r = max(5, s['peak_trains_per_hour'] * 0.8)
    folium.CircleMarker(
        [s['lat'], s['lon']], radius=r,
        color=LINE_COLORS.get(s['line'], '#888888'),
        fill=True, fill_color=LINE_COLORS.get(s['line'], '#888888'),
        fill_opacity=0.8, weight=2,
        tooltip=f"{s['station_name']} ({s['line']}) — {s['peak_trains_per_hour']}/hr peak",
    ).add_to(m)

# Layer 3 — venues
for _, v in wc_ref[wc_ref['type'].isin(['stadium', 'fan_fest'])].iterrows():
    folium.Marker([v['lat'], v['lon']], tooltip=v['name'],
                  icon=folium.Icon(color='red', icon='star')).add_to(m)
for _, h in wc_ref[wc_ref['type'] == 'hotel_cluster'].iterrows():
    folium.Marker([h['lat'], h['lon']], tooltip=h['name'],
                  icon=folium.Icon(color='purple', icon='bed', prefix='fa')).add_to(m)

# Save the map as a real artifact so it's also usable outside the notebook.
m.save(str(CHECKPOINTS / 'marta_map.html'))
print(f'✅ Map saved to {CHECKPOINTS / "marta_map.html"}')
m


### 🎯 What to look for on the map

Zoom into the stadium area (southwest of downtown). You should be
able to spot:

- **Two stations within half a kilometer** of the stadium: SEC District
  and Vine City. (SEC District was GWCC/CNN Center until a recent
  rename.)
- **Five Points** — the big hub just north of the stadium, where all
  four lines converge.
- **The hotel clusters** in Downtown, Midtown, and Buckhead — all on
  or near rail lines.

**Key insight:** The rail network already reaches the stadium well.
The question isn't whether the routes exist; it's whether the trains
can carry enough people. That's tomorrow's (Day 4) problem.


## 🔄 Recovery point — end of Section 2

If the map cell failed, run the next cell. It skips the map and just
reloads what Section 3 needs.


In [ ]:
ridership = pd.read_csv(
    CHECKPOINTS / 'day2_output_cleaned_ridership.csv', parse_dates=['date']
)
station_freq = pd.read_csv(CHECKPOINTS / 'day2_output_station_freq_clean.csv')
wc_ref = pd.read_csv(PROCESSED / 'worldcup_reference.csv')
print('✅ Recovery loaded.')


## 4. Section 3 — Venue-to-station distances

We need a concrete answer to: *how far is each venue from the nearest
rail station?* For that we use **great-circle distance** — the shortest
path along the Earth's surface between two points.

`geopy.distance.great_circle((lat1, lon1), (lat2, lon2)).km` returns
kilometers.


In [ ]:
import geopy.distance

venues = wc_ref[wc_ref['type'].isin(['stadium', 'fan_fest', 'hotel_cluster'])]

# 🎯 Your Turn!
# For each venue, find the *nearest* station and print it with the distance
# in km. The result for Mercedes-Benz Stadium should be around 0.32 km
# (SEC District Station).
#
# Hint: outer loop over venues, inner loop over station_freq; track the
# smallest distance and the station_name that produced it.
print('=== Venue → Nearest Station ===')
for _, v in venues.iterrows():
    best_d, best_n = float('inf'), ''
    for _, s in station_freq.iterrows():
        d = ???
        if d < best_d:
            best_d, best_n = d, s['station_name']
    print(f"  {v['name']:45s} → {best_n:30s} ({best_d:.2f} km)")


<details><summary>💡 Click for solution</summary>

```python
d = geopy.distance.great_circle(
    (v['lat'], v['lon']), (s['lat'], s['lon'])
).km
```

The inner loop compares every station to every venue — 38 × 5 = 190
comparisons. Python handles that instantly, but if we had 10,000 stops
(bus + rail) we'd want a vectorized approach. For this dataset, the
naive loop is fine and it's readable.
</details>


### ✅ Checkpoint — Day 3 done

Expected output above should include:

- Mercedes-Benz Stadium → **SEC District Station** (~0.32 km)
- Centennial Olympic Park → **Peachtree Center** (~0.4 km)
- Hotel clusters → each within ~1 km of a station

If these roughly match, you've got what Day 4 needs.


## What's next

**Day 4 — the main event.** You'll pull commute data from the Census
API, then build a 3×3 demand-vs-capacity grid to actually answer
*"Can MARTA handle the World Cup?"*. The EDA summary and station
frequencies you built this week feed directly into tomorrow's model.

Save this notebook (File → Save) and see you at 2 PM tomorrow.
